# 01 - Exploratory Data Analysis

Explores the synthetic inpatient encounter dataset produced by
`src/data_pipeline/ingest.py` (a schema-conformant stand-in for the
Azure Data Factory-landed `raw` zone, see `docs/architecture.md`).

Goals:
- Understand the overall readmission rate and class balance
- Inspect distributions of key clinical/utilization variables
- Look at univariate relationships with the `readmitted_30d` label
  to motivate the feature engineering choices in `docs/ml-design.md`

In [ ]:
import sys
from datetime import date
from pathlib import Path

# Allow imports from the repo root when running notebooks in-place.
sys.path.append(str(Path.cwd().parent.parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.data_pipeline.ingest import generate_synthetic_encounters

pd.set_option("display.max_columns", None)

In [ ]:
raw = generate_synthetic_encounters(n_rows=5000, start_date=date(2024, 1, 1), seed=42)
raw.shape

In [ ]:
raw.head()

In [ ]:
raw.describe(include="all").T

## Label balance

The 30-day readmission label is imbalanced, which motivates the
`scale_pos_weight` balancing in `src/ml_pipeline/train.py` and the
use of PR-AUC (alongside ROC-AUC) in `src/ml_pipeline/evaluate.py`.

In [ ]:
readmit_rate = raw["readmitted_30d"].mean()
print(f"Overall 30-day readmission rate: {readmit_rate:.2%}")

raw["readmitted_30d"].value_counts(normalize=True).plot.bar(title="Label distribution")
plt.xlabel("readmitted_30d")
plt.ylabel("proportion")
plt.show()

## Distributions of key numeric variables

In [ ]:
numeric_cols = [
    "age",
    "length_of_stay",
    "comorbidity_count",
    "charlson_index",
    "prior_admissions_12mo",
    "prior_ed_visits_12mo",
    "num_medications",
    "bmi",
    "systolic_bp",
    "glucose_level",
    "creatinine",
]

fig, axes = plt.subplots(4, 3, figsize=(14, 12))
for ax, col in zip(axes.ravel(), numeric_cols):
    raw[col].hist(ax=ax, bins=30)
    ax.set_title(col)
fig.tight_layout()
plt.show()

## Categorical breakdowns

In [ ]:
categorical_cols = ["sex", "ethnicity", "insurance_type", "admission_type", "discharge_disposition"]
for col in categorical_cols:
    print(raw[col].value_counts(normalize=True).round(3))
    print()

## Readmission rate by key risk drivers

Sanity-check that the synthetic data generator produces the expected
directional relationships described in `docs/ml-design.md` §2
(higher prior utilization, comorbidity burden, polypharmacy, SNF
discharge, and emergency admission should all increase readmission risk).

In [ ]:
by_prior_admissions = raw.groupby("prior_admissions_12mo")["readmitted_30d"].mean()
by_prior_admissions.plot.bar(title="Readmission rate by prior admissions (12mo)")
plt.ylabel("readmission rate")
plt.show()

In [ ]:
for col in ["discharge_disposition", "admission_type"]:
    rates = raw.groupby(col)["readmitted_30d"].mean().sort_values(ascending=False)
    print(f"Readmission rate by {col}:")
    print(rates.round(3))
    print()

In [ ]:
polypharmacy = (raw["num_medications"] >= 5)
raw.groupby(polypharmacy)["readmitted_30d"].mean().rename("readmission_rate").to_frame()

## Next steps

Continue to `02_feature_prototyping.ipynb` to prototype the derived
features implemented in `src/data_pipeline/feature_engineering.py`.